In [24]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


In [25]:
# -----------------------------
# CONFIG
# -----------------------------
INPUT_CSV = "/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_crisis_with_rile.csv"

DV = "rile_norm"
POP_COL = "Populism"
GOV_COL = "prior_president"
FE_COL = "person"
YEAR_COL = "date"      # set to None if you don't have it

RANDOM_STATE = 42
CONF_THRESH = 0.60     # robustness: ideology confidence
EQUIV_BOUND = 0.05     # substantive null bound for RILE


In [26]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8", low_memory=False)

required = [DV, POP_COL, GOV_COL, FE_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=required).copy()

df[POP_COL] = pd.to_numeric(df[POP_COL], errors="coerce").astype(int)
df[GOV_COL] = pd.to_numeric(df[GOV_COL], errors="coerce").astype(int)

print("Rows after cleaning:", len(df))
print("\nPopulism distribution (full sample):")
print(df[POP_COL].value_counts())


Rows after cleaning: 71808

Populism distribution (full sample):
0    69947
1     1861
Name: Populism, dtype: int64


In [27]:
df_pop = df[df[POP_COL] == 1]
df_non = df[df[POP_COL] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)
print("Non-populist rows (before):", len(df_non))

df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=RANDOM_STATE
)

df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print("\nBalanced Populism counts:")
print(df_balanced[POP_COL].value_counts())


Populist rows: 1861
Non-populist rows (before): 69947

Balanced Populism counts:
1    1861
0    1861
Name: Populism, dtype: int64


In [28]:
import re
import numpy as np
import pandas as pd

# df_balanced assumed to exist from your earlier balancing step
# If not: df_balanced = pd.read_csv(...)

# --- Ensure election_year exists (derive from person like "Bush_1988") ---
if "election_year" not in df_balanced.columns:
    df_balanced["election_year"] = (
        df_balanced["person"].astype(str).str.extract(r"_(\d{4})$", expand=False)
    )
df_balanced["election_year"] = pd.to_numeric(df_balanced["election_year"], errors="coerce")

# --- Parse last name too (optional, but handy) ---
df_balanced["last_name"] = df_balanced["person"].astype(str).str.replace(r"_(\d{4})$", "", regex=True)

# --- Comprehensive mapping for US major-party presidential candidates from 1952 to 2020 ---
# (Covers the names I see in your regression output; includes common ones through 2020.)
CANDIDATE_MAP = {
    # 1952 / 1956
    "Eisenhower_1952": "Eisenhower_Dwight",
    "Stevenson_1952": "Stevenson_Adlai",
    "Eisenhower_1956": "Eisenhower_Dwight",
    "Stevenson_1956": "Stevenson_Adlai",

    # 1960 / 1964 / 1968
    "Kennedy_1960": "Kennedy_JohnF",
    "Nixon_1960": "Nixon_Richard",
    "Johnson_1964": "Johnson_LyndonB",
    # (If you have Goldwater_1964, add it here.)
    "Humphrey_1968": "Humphrey_Hubert",
    "Nixon_1968": "Nixon_Richard",

    # 1972 / 1976 / 1980
    "McGovern_1972": "McGovern_George",
    "Nixon_1972": "Nixon_Richard",
    "Ford_1976": "Ford_Gerald",
    "Carter_1976": "Carter_Jimmy",
    "Carter_1980": "Carter_Jimmy",
    "Reagan_1980": "Reagan_Ronald",

    # 1984 / 1988
    "Mondale_1984": "Mondale_Walter",
    "Reagan_1984": "Reagan_Ronald",
    "Dukakis_1988": "Dukakis_Michael",
    "Bush_1988": "Bush_GeorgeHW",

    # 1992 / 1996 / 2000
    "Bush_1992": "Bush_GeorgeHW",
    "Clinton_1992": "Clinton_Bill",
    "Dole_1996": "Dole_Bob",
    "Clinton_1996": "Clinton_Bill",
    "Gore_2000": "Gore_Al",

    # 2004 / 2008 / 2012
    "Bush_2004": "Bush_GeorgeW",
    "Kerry_2004": "Kerry_John",
    "McCain_2008": "McCain_John",
    "Obama_2008": "Obama_Barack",
    "Romney_2012": "Romney_Mitt",
    "Obama_2012": "Obama_Barack",

    # 2016 / 2020
    "Trump_2016": "Trump_Donald",
    "Clinton_2016": "Clinton_Hillary",
    "Trump_2020": "Trump_Donald",
    "Biden_2020": "Joe_Biden",
}

# --- Apply mapping; fallback to treating each 'person' as its own individual ---
df_balanced["individual"] = df_balanced["person"].map(CANDIDATE_MAP)

unmapped = df_balanced["individual"].isna()
print("Unmapped rows:", int(unmapped.sum()))
print("Unmapped 'person' values (up to 50):")
print(df_balanced.loc[unmapped, "person"].drop_duplicates().head(50).to_list())

# Fallback: if unmapped, keep person as unique individual to avoid wrong merges
df_balanced.loc[unmapped, "individual"] = df_balanced.loc[unmapped, "person"]




Unmapped rows: 0
Unmapped 'person' values (up to 50):
[]


In [29]:
# How many individuals have variation in prior_president?
var_counts = df_balanced.groupby("individual")["prior_president"].nunique().value_counts()
print(var_counts)

# Show individuals with variation (best for identification)
varying = df_balanced.groupby("individual")["prior_president"].nunique()
print("\nIndividuals with prior_president variation (nunique==2):")
print(varying[varying == 2].index.to_list())


1    16
2     8
Name: prior_president, dtype: int64

Individuals with prior_president variation (nunique==2):
['Bush_GeorgeHW', 'Carter_Jimmy', 'Clinton_Bill', 'Eisenhower_Dwight', 'Nixon_Richard', 'Obama_Barack', 'Reagan_Ronald', 'Trump_Donald']


In [30]:
print(df_balanced[DV].describe())

print("\nMean RILE by prior_president:")
print(df_balanced.groupby(GOV_COL)[DV].mean())

print("\nMean RILE by Populism:")
print(df_balanced.groupby(POP_COL)[DV].mean())


count    3722.000000
mean        0.432176
std         0.602505
min        -0.996512
25%         0.058352
50%         0.705540
75%         0.917604
max         0.999235
Name: rile_norm, dtype: float64

Mean RILE by prior_president:
prior_president
0    0.423285
1    0.455571
Name: rile_norm, dtype: float64

Mean RILE by Populism:
Populism
0    0.506709
1    0.357644
Name: rile_norm, dtype: float64


In [31]:
def print_effect(res, term, data, dv=DV, bound=EQUIV_BOUND):
    coef = res.params[term]
    se = res.bse[term]
    ci_low = coef - 1.96 * se
    ci_high = coef + 1.96 * se
    sd = data[dv].std(ddof=1)

    print(f"\nEffect: {term}")
    print(f"coef      = {coef:.6f}")
    print(f"SE        = {se:.6f}")
    print(f"95% CI    = [{ci_low:.6f}, {ci_high:.6f}]")
    print(f"coef / SD = {coef/sd:.6f}")
    print(f"CI within ±{bound}? → {(ci_low >= -bound) and (ci_high <= bound)}")


In [32]:
import statsmodels.formula.api as smf

# Drop any rows missing the essentials
df_reg = df_balanced.dropna(subset=["rile_norm", "Populism", "prior_president", "individual", "election_year"]).copy()

fe_indiv_year = smf.ols(
    "rile_norm ~ Populism + prior_president + C(individual) + C(election_year)",
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["individual"]}
)

print(fe_indiv_year.summary())


                            OLS Regression Results                            
Dep. Variable:              rile_norm   R-squared:                       0.087
Model:                            OLS   Adj. R-squared:                  0.079
Method:                 Least Squares   F-statistic:                     10.32
Date:                Mon, 04 May 2026   Prob (F-statistic):            0.00387
Time:                        09:33:16   Log-Likelihood:                -3225.2
No. Observations:                3722   AIC:                             6520.
Df Residuals:                    3687   BIC:                             6738.
Df Model:                          34                                         
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 42, but rank is 1
  warnings.warn('covariance of constraints does not have full '


In [33]:
coef = fe_indiv_year.params["prior_president"]
se = fe_indiv_year.bse["prior_president"]
ci_low = coef - 1.96 * se
ci_high = coef + 1.96 * se

sd = df_reg["rile_norm"].std(ddof=1)

EQUIV_BOUND = 0.05  # change if you want ±0.10 etc.

print("prior_president effect:")
print("coef      =", round(coef, 4))
print("SE        =", round(se, 4))
print("95% CI    =", (round(ci_low, 4), round(ci_high, 4)))
print("coef/SD   =", round(coef / sd, 4))
print(f"CI within ±{EQUIV_BOUND}? →", (ci_low >= -EQUIV_BOUND) and (ci_high <= EQUIV_BOUND))


prior_president effect:
coef      = -0.1246
SE        = 0.0419
95% CI    = (-0.2068, -0.0425)
coef/SD   = -0.2069
CI within ±0.05? → False


In [34]:
# Keep only individuals with within-person variation in prior_president
switchers = (
    df_reg.groupby("individual")["prior_president"]
    .nunique()
    .pipe(lambda s: s[s == 2].index)
)

df_switchers = df_reg[df_reg["individual"].isin(switchers)].copy()

fe_switchers = smf.ols(
    "rile_norm ~ Populism + prior_president + C(individual) + C(election_year)",
    data=df_switchers
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_switchers["individual"]}
)

print(fe_switchers.summary())


                            OLS Regression Results                            
Dep. Variable:              rile_norm   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     1.178
Date:                Mon, 04 May 2026   Prob (F-statistic):              0.314
Time:                        09:33:16   Log-Likelihood:                -1699.6
No. Observations:                2045   AIC:                             3435.
Df Residuals:                    2027   BIC:                             3536.
Df Model:                          17                                         
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 23, but rank is 1
  warnings.warn('covariance of constraints does not have full '


In [35]:
switchers = [
    "Bush_GeorgeHW", "Carter_Jimmy", "Clinton_Bill", "Eisenhower_Dwight",
    "Nixon_Richard", "Obama_Barack", "Reagan_Ronald", "Trump_Donald"
]

df_switch = df_reg[df_reg["individual"].isin(switchers)]

fe_switch = smf.ols(
    "rile_norm ~ Populism + prior_president + C(individual) + C(election_year)",
    data=df_switch
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_switch["individual"]}
)

print(fe_switch.summary())


/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 23, but rank is 1
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:              rile_norm   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     1.178
Date:                Mon, 04 May 2026   Prob (F-statistic):              0.314
Time:                        09:33:16   Log-Likelihood:                -1699.6
No. Observations:                2045   AIC:                             3435.
Df Residuals:                    2027   BIC:                             3536.
Df Model:                          17                                         
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

In [36]:
cols = ["left_mass", "right_mass", "rile_raw", "rile_norm"]

examples = []

for col in cols:
    row = (
        df_balanced[df_balanced["Populism"] == 1]
        .sort_values(col, ascending=False)
        .iloc[0]
    )
    
    examples.append({
        "max_column": col,
        "max_value": row[col],
        "text": row["text"]
    })

examples_df = pd.DataFrame(examples)

display(examples_df)

,max_column,max_value,text
0,left_mass,0.995684,"Now, instead of gutting consumer protection, we should be expanding it. And we should build on the Dodd-Frank financial reforms and go even further because Wall Street can never, ever be permitted to threaten Main Street again. And the Wells Fargo scandal sheds light on another threat to consumers that we have to address. When the scam's victims, people like you and me, who had accounts there tried to sue, they were shocked to learn there was a provision in the very fine print of their contracts that kept them from going to court to sue the bank for being cheated. Instead, they are forced into a closed-door arbitration process without the important protections that you get in a court of law. We are not going to let corporations like Wells Fargo use these fine print """"""""""""""""gotchas"""""""""""""""" to escape accountability."
1,right_mass,0.992325,"So my case to the American people is this: At this time in our history, we simply cannot take the risk on a president with no national experience and a miserable Arkansas record to run on. Since I've been in the Oval Office, I've faced some very difficult decisions. That's what you pay me to do. And yes, I've made some mistakes. When I make a mistake, I'll admit it. But I believe I've been a good leader. I've tried to make the tough calls. I've tried to make the tough calls, willing to tell people not what they want to hear but what they need to hear. And I stand before you today asking for your support so that we can get to work with a new Congress to fix the problems that stand in the way of this country, and so that we reform our health care system, that we literally reinvent our schools, so that we can retrain workers from one generation and create jobs for the next, and so that we can cut government spending and cut taxes to get this economy moving again, and so that we can limit terms of members of the Congress and give government back to the people."
2,rile_raw,0.987434,"So my case to the American people is this: At this time in our history, we simply cannot take the risk on a president with no national experience and a miserable Arkansas record to run on. Since I've been in the Oval Office, I've faced some very difficult decisions. That's what you pay me to do. And yes, I've made some mistakes. When I make a mistake, I'll admit it. But I believe I've been a good leader. I've tried to make the tough calls. I've tried to make the tough calls, willing to tell people not what they want to hear but what they need to hear. And I stand before you today asking for your support so that we can get to work with a new Congress to fix the problems that stand in the way of this country, and so that we reform our health care system, that we literally reinvent our schools, so that we can retrain workers from one generation and create jobs for the next, and so that we can cut government spending and cut taxes to get this economy moving again, and so that we can limit terms of members of the Congress and give government back to the people."
3,rile_norm,0.994833,"Frankly, I'm fed up with politicians in Washington lecturing the rest of us about """"""""""""""""family values."""""""""""""""" Our families have values. But our government doesn't."


In [37]:
pd.set_option("display.max_colwidth", None)

display(
    examples_df.style
    .set_properties(subset=["text"], **{
        "white-space": "pre-wrap",
        "text-align": "left"
    })
    .set_properties(subset=["max_column", "max_value"], **{
        "text-align": "center"
    })
)

,max_column,max_value,text
0,left_mass,0.995684,"Now, instead of gutting consumer protection, we should be expanding it. And we should build on the Dodd-Frank financial reforms and go even further because Wall Street can never, ever be permitted to threaten Main Street again. And the Wells Fargo scandal sheds light on another threat to consumers that we have to address. When the scam's victims, people like you and me, who had accounts there tried to sue, they were shocked to learn there was a provision in the very fine print of their contracts that kept them from going to court to sue the bank for being cheated. Instead, they are forced into a closed-door arbitration process without the important protections that you get in a court of law. We are not going to let corporations like Wells Fargo use these fine print """"""""""""""""gotchas"""""""""""""""" to escape accountability."
1,right_mass,0.992325,"So my case to the American people is this: At this time in our history, we simply cannot take the risk on a president with no national experience and a miserable Arkansas record to run on. Since I've been in the Oval Office, I've faced some very difficult decisions. That's what you pay me to do. And yes, I've made some mistakes. When I make a mistake, I'll admit it. But I believe I've been a good leader. I've tried to make the tough calls. I've tried to make the tough calls, willing to tell people not what they want to hear but what they need to hear. And I stand before you today asking for your support so that we can get to work with a new Congress to fix the problems that stand in the way of this country, and so that we reform our health care system, that we literally reinvent our schools, so that we can retrain workers from one generation and create jobs for the next, and so that we can cut government spending and cut taxes to get this economy moving again, and so that we can limit terms of members of the Congress and give government back to the people."
2,rile_raw,0.987434,"So my case to the American people is this: At this time in our history, we simply cannot take the risk on a president with no national experience and a miserable Arkansas record to run on. Since I've been in the Oval Office, I've faced some very difficult decisions. That's what you pay me to do. And yes, I've made some mistakes. When I make a mistake, I'll admit it. But I believe I've been a good leader. I've tried to make the tough calls. I've tried to make the tough calls, willing to tell people not what they want to hear but what they need to hear. And I stand before you today asking for your support so that we can get to work with a new Congress to fix the problems that stand in the way of this country, and so that we reform our health care system, that we literally reinvent our schools, so that we can retrain workers from one generation and create jobs for the next, and so that we can cut government spending and cut taxes to get this economy moving again, and so that we can limit terms of members of the Congress and give government back to the people."
3,rile_norm,0.994833,"Frankly, I'm fed up with politicians in Washington lecturing the rest of us about """"""""""""""""family values."""""""""""""""" Our families have values. But our government doesn't."


In [38]:
cols = ["left_mass", "right_mass", "rile_raw", "rile_norm"]

examples_min = []

for col in cols:
    row = (
        df_balanced[df_balanced["Populism"] == 1]
        .sort_values(col, ascending=True)
        .iloc[0]
    )
    
    examples_min.append({
        "min_column": col,
        "min_value": row[col],
        "text": row["text"]
    })

examples_min_df = pd.DataFrame(examples_min)

display(examples_min_df)

,min_column,min_value,text
0,left_mass,0.000471,My ethics plan will end corruption in our government. We will. Corruption is massive.
1,right_mass,0.001748,My ethics plan will end corruption in our government. We will. Corruption is massive.
2,rile_raw,-0.993799,"Now, instead of gutting consumer protection, we should be expanding it. And we should build on the Dodd-Frank financial reforms and go even further because Wall Street can never, ever be permitted to threaten Main Street again. And the Wells Fargo scandal sheds light on another threat to consumers that we have to address. When the scam's victims, people like you and me, who had accounts there tried to sue, they were shocked to learn there was a provision in the very fine print of their contracts that kept them from going to court to sue the bank for being cheated. Instead, they are forced into a closed-door arbitration process without the important protections that you get in a court of law. We are not going to let corporations like Wells Fargo use these fine print """"""""""""""""gotchas"""""""""""""""" to escape accountability."
3,rile_norm,-0.996222,"Now, instead of gutting consumer protection, we should be expanding it. And we should build on the Dodd-Frank financial reforms and go even further because Wall Street can never, ever be permitted to threaten Main Street again. And the Wells Fargo scandal sheds light on another threat to consumers that we have to address. When the scam's victims, people like you and me, who had accounts there tried to sue, they were shocked to learn there was a provision in the very fine print of their contracts that kept them from going to court to sue the bank for being cheated. Instead, they are forced into a closed-door arbitration process without the important protections that you get in a court of law. We are not going to let corporations like Wells Fargo use these fine print """"""""""""""""gotchas"""""""""""""""" to escape accountability."


In [39]:
cols = ["left_mass", "right_mass", "rile_raw", "rile_norm"]

examples_median = []

df_pop = df_balanced[df_balanced["Populism"] == 1].copy()

for col in cols:
    median_val = df_pop[col].median()
    
    # Pick the row whose value is closest to the median
    row = df_pop.loc[(df_pop[col] - median_val).abs().idxmin()]
    
    examples_median.append({
        "median_column": col,
        "median_value": median_val,
        "example_value": row[col],
        "text": row["text"]
    })

examples_median_df = pd.DataFrame(examples_median)

display(examples_median_df)

,median_column,median_value,example_value,text
0,left_mass,0.082987,0.082987,"Your products will be sent around the world, not your jobs. Hillary is controlled by special interests who want to ship your jobs to other countries. She's the most corrupt person ever to run for President."
1,right_mass,0.380005,0.380005,"And yes, truth over lies. So it's time to stand up and take back our democracy. May God bless you."
2,rile_raw,0.203986,0.203986,Drain the swamp. Drain the swamp. Is there any place better to be than a Trump rally? Are we having fun?
3,rile_norm,0.563432,0.563432,"And now, after twenty-one months and three debates, Senator McCain still has not been able to tell the American people a single major thing he'd do differently from George Bush when it comes to the economy. Senator McCain says that we can't spend the next four years waiting for our luck to change, but you understand that the biggest gamble we can take is embracing the same old Bush-McCain policies that have failed us for the last eight years. It's not change when John McCain wants to give a $700,000 tax cut to the average Fortune 500 CEO. It's not change when he wants to give $200 billion to the biggest corporations or $4 billion to the oil companies or $300 billion to the same Wall Street banks that got us into this mess. It's not change when he comes up with a tax plan that doesn't give a penny of relief to more than 100 million middle-class Americans."


In [40]:
cols = ["left_mass", "right_mass", "rile_norm"]

df_pop = df_balanced[df_balanced["Populism"] == 1].copy()
df_nonpop = df_balanced[df_balanced["Populism"] == 0].copy()

for col in cols:
    print("\n" + "="*80)
    print(f"MEDIAN EXAMPLES FOR: {col}")
    print("="*80)
    
    for label, df_group in [("Non-populist", df_nonpop), ("Populist", df_pop)]:
        median_val = df_group[col].median()
        row = df_group.loc[(df_group[col] - median_val).abs().idxmin()]
        
        print(f"\n--- {label} ---")
        print(f"Median {col}: {median_val}")
        print(f"Example value: {row[col]}")
        print("\nText:")
        print(row["text"])


MEDIAN EXAMPLES FOR: left_mass

--- Non-populist ---
Median left_mass: 0.06293842
Example value: 0.06293842

Text:
I say that you want instead men who are working for progress and hope and human decency. We have great tasks ahead of us. We must continue with imagination and with implacable purpose the task of building peace in this tormented world. And here at home we must continue to strive for a better living for all of our people. We are united in a single purpose--the great American purpose of freeing men to use their strength and their courage in building a better life. This is the purpose for which the Democratic party stands, and this, my friends, is the purpose I should like to serve with your help.

--- Populist ---
Median left_mass: 0.0829873
Example value: 0.0829873

Text:
Your products will be sent around the world, not your jobs. Hillary is controlled by special interests who want to ship your jobs to other countries. She's the most corrupt person ever to run for Presiden

In [41]:
cols = ["left_mass", "right_mass", "rile_raw", "rile_norm"]

df_nonpop = df_balanced[df_balanced["Populism"] == 0].copy()

nonpop_minmax_examples = []

for col in cols:
    min_row = df_nonpop.sort_values(col, ascending=True).iloc[0]
    max_row = df_nonpop.sort_values(col, ascending=False).iloc[0]
    
    nonpop_minmax_examples.append({
        "column": col,
        "type": "min",
        "value": min_row[col],
        "text": min_row["text"]
    })
    
    nonpop_minmax_examples.append({
        "column": col,
        "type": "max",
        "value": max_row[col],
        "text": max_row["text"]
    })

nonpop_minmax_examples_df = pd.DataFrame(nonpop_minmax_examples)

pd.set_option("display.max_colwidth", None)

display(
    nonpop_minmax_examples_df.style
    .set_properties(subset=["text"], **{
        "white-space": "pre-wrap",
        "text-align": "left"
    })
    .set_properties(subset=["column", "type", "value"], **{
        "text-align": "center"
    })
)

,column,type,value,text
0,left_mass,min,0.000382,This is the faith teaching us all that we are children of God. It teaches us the divine origin of each man's dignity. It teaches us the sublime meaning of our brotherhood under His fatherhood.
1,left_mass,max,0.996103,"All of us need to understand that perhaps our greatest deficiency today is in the teaching of reading. Moreover, despite the current emphasis on preschool training, some one-third of our schools do not have kindergartens. I keenly feel these shortcomings and hope to play an important part in their correction."
2,right_mass,min,0.001398,"Listen, we've got to do more to help our workers gain the skills necessary to fill the jobs of the 21st century. That's why I know we need to double the number of people served by our job training programs and increase funding for our community college systems. One other issue that's important, in terms of education, is that most new jobs are filled by people with at least 2 years of college, yet, one in four students gets there. That's why I believe we need early intervention programs to help students in high school. We want everybody to have the skills necessary to move on. We'll place a new focus on math and science in our high schools. Over time, we'll require a rigorous exam before graduation. By raising performance in our high schools and by expanding Pell grants for low-income and middle-income families, we will help more Americans start their career with a college diploma."
3,right_mass,max,0.998234,"The seventh, a federal death penalty. I think certain acts of violence deserve the ultimate penalty. I'm talking about assassinations, murder for hire, terrorism, and other depraved acts. Add to that the new urban violence we see with gangs, drive-by shootings, random violence, gang massacres. These people are merchants of death, who trade in death. The death penalty is warranted in these cases. And I wish Congress would move and do something about it."
4,rile_raw,min,-0.993928,"All of us need to understand that perhaps our greatest deficiency today is in the teaching of reading. Moreover, despite the current emphasis on preschool training, some one-third of our schools do not have kindergartens. I keenly feel these shortcomings and hope to play an important part in their correction."
5,rile_raw,max,0.997727,This is the faith teaching us all that we are children of God. It teaches us the divine origin of each man's dignity. It teaches us the sublime meaning of our brotherhood under His fatherhood.
6,rile_norm,min,-0.996512,"And every adult should have access to a community education center to be reeducated. The average eighteen-year-old will change jobs eight times in a lifetime. We need education for the forty and the fifty and the sixty-year-olds, too, so that they will not be dragged down by the global economy, but be lifted up."
7,rile_norm,max,0.999235,This is the faith teaching us all that we are children of God. It teaches us the divine origin of each man's dignity. It teaches us the sublime meaning of our brotherhood under His fatherhood.


In [42]:
df.columns

Index(['Speech_id', 'text', 'party', 'term', 'comp', 'populist_old_keywords',
       'par_id', 'speech_par_id', 'Campaign', 'Authoritarianism', 'Exclusion',
       'Inclusion', 'High_pride', 'Low_pride', 'Populism', 'party_incumbent',
       'recession', 'prior_president', 'election_date', 'person', 'date',
       'name_date', 'weeks_to_election', 'cutoff_date', 'crisis_count',
       'crisis_distinct', 'word_count', 'crisis_per_100w', 'election_year',
       'left_mass', 'right_mass', 'rile_raw', 'rile_norm', 'top_label',
       'top_prob'],
      dtype='object')

In [48]:
import pandas as pd


# The example texts from your LaTeX table
example_texts = {
    "left_mass_min_pop":    "My ethics plan will end corruption in our government. We will. Corruption is massive.",
    "left_mass_min_nonpop": "This is the faith teaching us all that we are children of God. It teaches us the divine origin of each man's dignity. It teaches us the sublime meaning of our brotherhood under His fatherhood.",
    "left_mass_med_pop":    "Your products will be sent around the world, not your jobs. Hillary is controlled by special interests who want to ship your jobs to other countries. She's the most corrupt person ever to run for President.",
    "left_mass_med_nonpop": "I say that you want instead men who are working for progress and hope and human decency.",  # use a distinctive substring
    "left_mass_max_pop":    "Now, instead of gutting consumer protection, we should be expanding it.",  # substring
    "left_mass_max_nonpop": "All of us need to understand that perhaps our greatest deficiency today is in the teaching of reading.",

    "right_mass_min_pop":   "My ethics plan will end corruption in our government. We will. Corruption is massive.",
    "right_mass_min_nonpop":"Listen, we've got to do more to help our workers gain the skills necessary to fill the jobs of the 21st century.",
    "right_mass_med_pop":   "And yes, truth over lies. So it's time to stand up and take back our democracy.",
    "right_mass_med_nonpop":"He served in the United States Senate for 20 years, and he's voted for higher taxes 98 times.",
    "right_mass_max_pop":   "So my case to the American people is this: At this time in our history, we simply cannot take the risk",
    "right_mass_max_nonpop":"The seventh, a federal death penalty.",

    "dir_min_pop":          "Now, instead of gutting consumer protection, we should be expanding it.",  # same as left_mass_max_pop
    "dir_min_nonpop":       "And every adult should have access to a community education center to be reeducated.",
    "dir_med_pop":          "And now, after twenty-one months and three debates, Senator McCain still has not been able to tell",
    "dir_med_nonpop":       "The past two days I've discussed the issues missing from the Carter campaign",
    "dir_max_pop":          "Frankly, I'm fed up with politicians in Washington lecturing the rest of us about",
    "dir_max_nonpop":       "This is the faith teaching us all that we are children of God.",  # same as left_mass_min_nonpop
}

# Look up each example using substring matching
results = []
for label, snippet in example_texts.items():
    # Use the first ~60 chars as a reliable substring
    search_str = snippet[:60]
    matches = df[df["text"].str.contains(search_str[:50], regex=False, na=False)]
    
    if len(matches) == 0:
        results.append({"label": label, "person": "NOT FOUND", "date": "NOT FOUND", "text_preview": snippet[:80]})
    elif len(matches) > 1:
        # If multiple hits, show all — you'll pick manually
        for _, row in matches.iterrows():
            results.append({
                "label": label,
                "person": row[FE_COL],
                "date": row[YEAR_COL] if YEAR_COL else "N/A",
                "n_matches": len(matches),
                "text_preview": row["text"][:120]
            })
    else:
        row = matches.iloc[0]
        results.append({
            "label": label,
            "person": row[FE_COL],
            "date": row[YEAR_COL] if YEAR_COL else "N/A",
            "n_matches": 1,
            "text_preview": row["text"][:120]
        })

pd.set_option("display.max_colwidth", None)
results_df = pd.DataFrame(results)
display(results_df[["label", "person", "date", "n_matches", "text_preview"]])

,label,person,date,n_matches,text_preview
0,left_mass_min_pop,Trump_2016,2016,1,My ethics plan will end corruption in our government. We will. Corruption is massive.
1,left_mass_min_nonpop,Eisenhower_1952,1952,1,This is the faith teaching us all that we are children of God. It teaches us the divine origin of each man's dignity.
2,left_mass_med_pop,Trump_2016,2016,1,"Your products will be sent around the world, not your jobs. Hillary is controlled by special interests who want to ship"
3,left_mass_med_nonpop,Stevenson_1952,1952,1,I say that you want instead men who are working for progress and hope and human decency. We have great tasks ahead of us
4,left_mass_max_pop,Clinton_2016,2016,1,"Now, instead of gutting consumer protection, we should be expanding it. And we should build on the Dodd-Frank financial"
5,left_mass_max_nonpop,Nixon_1968,1968,1,"All of us need to understand that perhaps our greatest deficiency today is in the teaching of reading. Moreover, despite"
6,right_mass_min_pop,Trump_2016,2016,1,My ethics plan will end corruption in our government. We will. Corruption is massive.
7,right_mass_min_nonpop,Bush_2004,2004,1,"Listen, we've got to do more to help our workers gain the skills necessary to fill the jobs of the 21st century. That's"
8,right_mass_med_pop,Biden_2020,2020,6,"And yes, truth over lies. So it's time to stand up and take back our democracy. We can do this."
9,right_mass_med_pop,Biden_2020,2020,6,"And yes, truth over lies. So it's time to stand up and take back our democracy. We can do this."
